
# 🚦 Parking Violation Regressor **(Hackathon Edition v10/v4)**
**Flipkart Grid Hackathon 2.0 — Round 2: ParkingObserver**

### Hackathon Architecture Updates:
Since external data (like OpenStreetMap) is forbidden, this notebook extracts internal context features mathematically to improve accuracy without data leakage.

1. **Temporal Splitting**: Training on historical data (Nov 2023 - Mar 2024). Testing on the holdout future month (April 2024). This prevents spatial data leakage.
2. **Internal Context Extraction**: Uses historical data from the training set to infer location characteristics:
   - `hist_violation_count`: Total historical density (differentiates highways from side streets).
   - `temporal_cluster_id`: K-Means clustering on 24-hour traffic profiles to automatically categorize locations.
3. **Business Value Metrics**: Evaluates model performance using **Cumulative Lift (ROI)**. Regression metrics (like R^2) fail on stochastic human behavior, but ranking metrics prove the model's logistical value.


## 0. Colab Setup & Dependencies

In [ ]:

# Run once if packages are missing in Colab
!pip install catboost scikit-learn pandas numpy matplotlib seaborn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.0 MB/s eta 0:00:00


In [ ]:

# Optional: Mount Google Drive if your dataset is stored there
# from google.colab import drive
# drive.mount('/content/drive')


## 1. Imports & Seed

In [ ]:

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path('jan to may police violation_anonymized791b166.csv')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

print('All imports successful.')
print(f'Data path: {DATA_PATH.resolve()}')


All imports successful.
Data path: /content/jan to may police violation_anonymized791b166.csv


## 2. Load & Clean Data

In [ ]:

# Load Data
print("Loading data...")
df = pd.read_csv(DATA_PATH, encoding='utf-8', on_bad_lines='skip', low_memory=False)

# Basic Extraction
pcu_map = {
    'SCOOTER': 0.5, 'MOTOR CYCLE': 0.5, 'MOPED': 0.5,
    'CAR': 1.0, 'JEEP': 1.0,
    'PASSENGER AUTO': 1.5, 'GOODS AUTO': 1.5, 'VAN': 1.5, 'TEMPO': 1.5, 'MAXI-CAB': 1.5,
    'PRIVATE BUS': 3.0, 'BUS (BMTC/KSRTC)': 3.0, 'TOURIST BUS': 3.0, 'SCHOOL VEHICLE': 3.0,
    'FACTORY BUS': 3.0, 'HGV': 3.0, 'LORRY/GOODS VEHICLE': 3.0, 'LGV': 3.0, 'TANKER': 3.0,
    'MINI LORRY': 3.0, 'TRACTOR': 3.0, 'OTHERS': 1.0, 'UNKNOWN': 1.0
}
df['vehicle_type_clean'] = df['updated_vehicle_type'].fillna(df['vehicle_type']).fillna('UNKNOWN').astype(str).str.upper()
df['pcu_weight'] = df['vehicle_type_clean'].map(pcu_map).fillna(1.0)

df['created_datetime'] = pd.to_datetime(df['created_datetime'], utc=True, errors='coerce')
df['modified_datetime'] = pd.to_datetime(df['modified_datetime'], utc=True, errors='coerce')
df['modified_datetime'] = df['modified_datetime'].fillna(df['created_datetime'] + pd.Timedelta(minutes=30))
df['duration_hours'] = (df['modified_datetime'] - df['created_datetime']).dt.total_seconds() / 3600.0
df['duration_hours'] = df['duration_hours'].clip(0, 24)

df['junction_name'] = df['junction_name'].fillna('Unknown').astype(str)
df['is_junction'] = ((df['junction_name'] != 'Unknown') & (df['junction_name'] != '')).astype(int)
df['traffic_impact_score'] = df['duration_hours'] * df['pcu_weight'] * (1 + df['is_junction'])

df['hour_of_day'] = df['created_datetime'].dt.hour.astype('Int16')
df['lat_bin_broad'] = df['latitude'].round(3).astype(str)
df['lng_bin_broad'] = df['longitude'].round(3).astype(str)
df['loc_key_broad'] = df['lat_bin_broad'] + '_' + df['lng_bin_broad']


Loading data...


## 3. Temporal Split

In [ ]:

# Temporal Split
split_date = pd.to_datetime('2024-04-01', utc=True)
df_train = df[df['created_datetime'] < split_date].copy()
df_test = df[df['created_datetime'] >= split_date].copy()


## 4. Internal Context Extraction

In [ ]:

# Internal Context Extraction (ONLY ON TRAINING SET)
print("Extracting Internal Context (K-Means & Density)...")
hist_density = df_train.groupby('loc_key_broad').size().rename('hist_violation_count')
hist_junction = df_train.groupby('loc_key_broad')['is_junction'].mean().rename('hist_junction_ratio')

profile = df_train.groupby(['loc_key_broad', 'hour_of_day']).size().unstack(fill_value=0)
profile_norm = profile.div(profile.sum(axis=1), axis=0).fillna(0)

kmeans = KMeans(n_clusters=4, random_state=SEED)
profile['temporal_cluster_id'] = kmeans.fit_predict(profile_norm)
cluster_map = profile['temporal_cluster_id']

context_df = pd.concat([hist_density, hist_junction, cluster_map], axis=1).fillna(0).reset_index()


Extracting Internal Context (K-Means & Density)...


## 5. Grid Generation & Aggregation

In [ ]:

# Data Aggregation & Zero-Filling Pipeline
def build_grid(base_df, context_df):
    valid = base_df.dropna(subset=['hour_of_day', 'loc_key_broad', 'traffic_impact_score']).copy()
    valid['day_of_week'] = valid['created_datetime'].dt.dayofweek.astype('Int16')
    valid['is_weekend'] = (valid['day_of_week'] >= 5).astype('Int16')

    agg = valid.groupby(['loc_key_broad', 'day_of_week', 'is_weekend', 'hour_of_day']).agg(
        total_impact=('traffic_impact_score', 'sum'),
        violation_count=('id', 'count')
    ).reset_index()

    import itertools
    unique_locs = valid['loc_key_broad'].unique()
    grid = pd.DataFrame(list(itertools.product(unique_locs, range(7), range(24))), columns=['loc_key_broad', 'day_of_week', 'hour_of_day'])
    grid['is_weekend'] = (grid['day_of_week'] >= 5).astype('Int16')
    grid['hour_of_day'] = grid['hour_of_day'].astype('Int16')
    grid['day_of_week'] = grid['day_of_week'].astype('Int16')

    final = pd.merge(grid, agg, on=['loc_key_broad', 'day_of_week', 'is_weekend', 'hour_of_day'], how='left')
    final['total_impact'] = final['total_impact'].fillna(0.0)
    final['violation_count'] = final['violation_count'].fillna(0.0)

    final = pd.merge(final, context_df, on='loc_key_broad', how='left')
    final['hist_violation_count'] = final['hist_violation_count'].fillna(0)
    final['hist_junction_ratio'] = final['hist_junction_ratio'].fillna(0)
    final['temporal_cluster_id'] = final['temporal_cluster_id'].fillna(4).astype('Int16') # 4 = Unknown

    final[['lat_bin', 'lng_bin']] = final['loc_key_broad'].str.split('_', expand=True).astype(float)
    return final

print("Building Grids...")
train_df = build_grid(df_train, context_df)
test_df = build_grid(df_test, context_df)


Building Grids...


## 6. Target Capping

In [ ]:

# Target Capping
p99 = train_df['total_impact'].quantile(0.99)
train_df['total_impact'] = train_df['total_impact'].clip(upper=p99)
test_df['total_impact'] = test_df['total_impact'].clip(upper=p99)

features = ['lat_bin', 'lng_bin', 'hour_of_day', 'day_of_week', 'is_weekend', 'hist_violation_count', 'hist_junction_ratio', 'temporal_cluster_id']
cat_features = ['hour_of_day', 'day_of_week', 'is_weekend', 'temporal_cluster_id']

X_train = train_df[features]
y_train = train_df['total_impact']
X_test = test_df[features]
y_test = test_df['total_impact']


## 7. Model Training (Two-Stage Hurdle)

In [ ]:

# Train Hurdle Model
print("Training Stage 1 Classifier...")
y_train_clf = (y_train > 0).astype(int)
clf = CatBoostClassifier(iterations=1200, depth=6, learning_rate=0.05, loss_function='Logloss', cat_features=cat_features, random_seed=SEED, verbose=100)
clf.fit(Pool(X_train, y_train_clf, cat_features=cat_features))

print("Training Stage 2 Regressor...")
pos_mask = y_train > 0
reg = CatBoostRegressor(iterations=1300, depth=7, learning_rate=0.03, loss_function='RMSE', cat_features=cat_features, random_seed=SEED, verbose=100)
reg.fit(Pool(X_train[pos_mask], y_train[pos_mask], cat_features=cat_features))

print("Running Inference on April Test Set...")
test_probs = clf.predict_proba(X_test)[:, 1]
test_preds_reg = np.clip(reg.predict(X_test), 0, None)
test_preds = test_probs * test_preds_reg

test_df['predicted_impact'] = test_preds


Training Stage 1 Classifier...
0:	learn: 0.6153668	total: 1.24s	remaining: 24m 46s
100:	learn: 0.1254331	total: 2m 25s	remaining: 26m 25s
200:	learn: 0.1234664	total: 4m 49s	remaining: 23m 56s
300:	learn: 0.1226995	total: 7m 13s	remaining: 21m 35s
400:	learn: 0.1221494	total: 9m 50s	remaining: 19m 37s
500:	learn: 0.1216725	total: 12m 20s	remaining: 17m 12s
600:	learn: 0.1212373	total: 14m 59s	remaining: 14m 56s
700:	learn: 0.1208419	total: 17m 59s	remaining: 12m 48s
800:	learn: 0.1205208	total: 20m 41s	remaining: 10m 18s
900:	learn: 0.1201796	total: 23m 21s	remaining: 7m 45s
1000:	learn: 0.1198512	total: 25m 59s	remaining: 5m 9s
1100:	learn: 0.1195769	total: 28m 34s	remaining: 2m 34s
1199:	learn: 0.1193292	total: 31m 4s	remaining: 0us
Training Stage 2 Regressor...
0:	learn: 18.2812505	total: 62.2ms	remaining: 1m 20s
100:	learn: 17.6144549	total: 6.54s	remaining: 1m 17s
200:	learn: 17.4463089	total: 11.4s	remaining: 1m 2s
300:	learn: 17.3403499	total: 15.7s	remaining: 52.1s
400:	learn: 

## 8. Business Metrics & Evaluation

In [ ]:

# Metrics
rmse = np.sqrt(mean_squared_error(y_test, test_preds))
r2 = r2_score(y_test, test_preds)
print(f"\nModel RMSE: {rmse:.4f}")
print(f"Model R2: {r2:.4f}")

# 1. Top-100 Precision
true_top_100 = test_df.nlargest(100, 'total_impact')
pred_top_100 = test_df.nlargest(100, 'predicted_impact')
hits = len(set(true_top_100.index).intersection(set(pred_top_100.index)))
print(f"\n=== TOP-100 CHOKE POINT PRECISION ===")
print(f"Out of the true Top 100 worst intersections in April, the model successfully identified {hits}.")
print(f"Accuracy at identifying extreme choke points: {hits}%")

# 2. Cumulative Lift Curve Stats
test_df = test_df.sort_values(by='predicted_impact', ascending=False)
total_impact_citywide = test_df['total_impact'].sum()

print("\n=== CUMULATIVE LIFT (RESOURCE ALLOCATION ROI) ===")
for pct in [1, 5, 10, 20]:
    k = int(len(test_df) * (pct / 100.0))
    captured_impact = test_df.head(k)['total_impact'].sum()
    lift = (captured_impact / total_impact_citywide) * 100
    print(f"If Flipkart monitors the Top {pct}% of predicted zones, they intercept {lift:.1f}% of all actual traffic impact.")



Model RMSE: 4.6666
Model R2: -2.4032

=== TOP-100 CHOKE POINT PRECISION ===
Out of the true Top 100 worst intersections in April, the model successfully identified 0.
Accuracy at identifying extreme choke points: 0%

=== CUMULATIVE LIFT (RESOURCE ALLOCATION ROI) ===
If Flipkart monitors the Top 1% of predicted zones, they intercept 15.6% of all actual traffic impact.
If Flipkart monitors the Top 5% of predicted zones, they intercept 37.0% of all actual traffic impact.
If Flipkart monitors the Top 10% of predicted zones, they intercept 50.4% of all actual traffic impact.
If Flipkart monitors the Top 20% of predicted zones, they intercept 66.7% of all actual traffic impact.


## 9. Export Models

In [ ]:

import pickle

print("Saving models to disk...")
pkl_path = MODEL_DIR / 'v10_hackathon_model.pkl'

with open(pkl_path, 'wb') as f:
    pickle.dump({
        'clf': clf,
        'reg': reg,
        'kmeans': kmeans,
        'features': features,
        'cat_features': cat_features,
        'rmse': rmse,
        'r2': r2
    }, f)

print(f"Model dictionary saved successfully to {pkl_path}!")


Saving models to disk...
Model dictionary saved successfully to models/v10_hackathon_model.pkl!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
